In [1]:
!python -c "import torch; print('CUDA available:', torch.cuda.is_available())"

CUDA available: True


In [3]:
!python dkt_preprocess_merged.py \
  --input merged_student_question_history.csv \
  --output_dir dkt_processed \
  --min_seq_len 3 \
  --train_ratio 0.8 --valid_ratio 0.1 --test_ratio 0.1

Reading input CSV from merged_student_question_history.csv ...
Detecting question indices from qN_id columns ...
Found 364 question slots: from q1 to q364
Extracting sequences per user (this may take a moment) ...
  processed 500 rows ...
  processed 1000 rows ...
  processed 1500 rows ...
Kept 1590 users with seq_len >= 3.
Building question id to index mapping ...
Number of unique questions: 95
Saving train split to dkt_processed/dkt_train.pkl ...
Saving valid split to dkt_processed/dkt_valid.pkl ...
Saving test split to dkt_processed/dkt_test.pkl ...
Saving metadata to dkt_processed/dkt_metadata.json ...
Saving question id mapping to dkt_processed/dkt_question_mapping.json ...
Done.


In [15]:
!python train_dkt_merged.py \
  --data_dir dkt_processed \
  --max_len 300 \
  --hidden 200 \
  --dropout 0.1 \
  --batch_size 64 \
  --epochs 50 \
  --lr 1e-3 \
  --device cuda

Using device: cuda
Metadata: num_questions=95, total_q_slots=364.0
Epoch 1/50
  [Train] loss=0.5667, AUC=0.6324, ACC=0.7457
  Fairness per completion-rate bin:
    Bin 1 (10- 20%): count=3306, TPR=0.990, FPR=0.922, ACC=0.774
    Bin 2 (20- 30%): count=97, TPR=0.988, FPR=0.833, ACC=0.887
    Bin 3 (30- 40%): count=535, TPR=0.981, FPR=0.811, ACC=0.817
    Bin 4 (40- 50%): count=8490, TPR=0.973, FPR=0.781, ACC=0.848
    Bin 5 (50- 60%): count=2188, TPR=0.974, FPR=0.758, ACC=0.840
    Bin 6 (60- 70%): count=2788, TPR=0.976, FPR=0.823, ACC=0.817
    Bin 7 (70- 80%): count=1602, TPR=0.969, FPR=0.827, ACC=0.759
    Bin 8 (80- 90%): count=293, TPR=0.954, FPR=0.816, ACC=0.754
    Bin 9 (90-100%): count=897, TPR=0.988, FPR=0.783, ACC=0.810
  Equalized odds distance (lowest vs highest non-empty bin): 0.1395
  Accuracy variance across bins: 0.001694
  [Valid] loss=0.4342, AUC=0.7349, ACC=0.8199
          EO(low-high)=0.1395, ACC var=0.001694
  [Info] New best model saved to dkt_best.pt (valid AUC=